# A1.8 · Model routing architecture

**Function A — Security Architecture & Platform → The Security Architect**  ·  *Both directions*

Builds on **[A1.7 · Build vs buy](https://spbreed.github.io/cyber-commons/lessons/A1.7.html)**.

| | |
|---|---|
| Open-source tooling | LiteLLM, vLLM |
| Open-weight models | GLM-4.6, Llama 3.3, Kimi K2 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


Routing between models is presented as a cost decision: use the big model where
it matters, the small one everywhere else. Real deployments do exactly that, and
the saving is genuine.

The security content is in *which stage gets which model*, because the stages
have very different relationships to authority:

- **Plan** — reads context, produces a strategy. Touches nothing.
- **Act** — chooses and invokes tools. This is where authority lives.
- **Verify** — decides whether the result is acceptable. This is where *trust*
  lives.

The common optimisation puts the large model on planning (it looks impressive in
demos) and a small fast model on acting, so the loop stays responsive. That
places the weakest reasoning next to the highest authority.

The second common optimisation puts the cheapest model on verification, because
verification "is just a check". That is worse: a weak verifier does not fail
loudly, it *approves*. B2.2 is an entire lesson on why.

## 2 · Demo — the routing table as a security artefact

Open-weight models, since this curriculum assumes no frontier account: Kimi K2 and GLM for heavy reasoning, Llama 3.3 in its smaller sizes for the fast paths. The table below is the deliverable — it says which model may trigger what.

In [ ]:
SCOPE_WEIGHT = {"self": 1, "project": 3, "tenant": 8, "org": 20}

ROUTES = {
 #  stage      model                  tools it may invoke               gated
 "plan":   ("Kimi K2 (large, open)",  [],                                set()),
 "act":    ("GLM-4.6 (mid, open)",    [("write_file", "project", True),
                                       ("open_pr",    "project", True)], {"open_pr"}),
 "verify": ("GLM-4.6 (mid, open)",    [],                                set()),
 "summarise": ("Llama 3.3 8B (small)", [],                               set()),
}
def stage_blast(tools, gated):
    return sum(SCOPE_WEIGHT[s] * (1 if rev else 2)
               for n, s, rev in tools if n not in gated)

print(f"{'stage':11s}{'model':26s}{'tools':>6}{'blast':>7}")
print("-" * 52)
for stage, (model, tools, gated) in ROUTES.items():
    print(f"{stage:11s}{model:26s}{len(tools):>6}{stage_blast(tools, gated):>7}")
print("\nOnly one stage holds tools at all, and its one risky tool is gated.")

## 3 · Where it breaks — the two cheap optimisations

Both of these get proposed in every performance review of an agent system, and both are reasonable-sounding.

In [ ]:
def evaluate(routes, label):
    total = sum(stage_blast(t, g) for _, t, g in routes.values())
    actor = [(s, m) for s, (m, t, g) in routes.items() if t]
    verifier = routes["verify"][0]
    print(f"{label}")
    print(f"   system blast {total:3d}   tools held by: {actor}")
    print(f"   verifier model: {verifier}")
    return total

good = evaluate(ROUTES, "as designed")

# optimisation 1: move tools to the fast small model so the loop feels snappy
opt1 = dict(ROUTES)
opt1["act"] = ("Llama 3.3 8B (small)",
               [("write_file", "project", True), ("open_pr", "project", True),
                ("deploy", "org", False)], set())
bad1 = evaluate(opt1, "\n'speed up the act stage' — small model, ungated, +deploy")

# optimisation 2: verify with the cheapest thing available
opt2 = dict(ROUTES)
opt2["verify"] = ("Llama 3.2 1B (tiny)", [], set())
evaluate(opt2, "\n'verification is just a check' — tiny model verifies")
print("   blast unchanged — and that is exactly why this one is dangerous:")
print("   the metric does not move, but every result is now trusted on the")
print("   word of the weakest model in the system. See B2.2.")

## 4 · The control — two routing rules

State them as rules a policy engine can enforce, not as guidance:

1. **Capability decides who plans. Blast radius decides who acts.** A model may only hold tools whose combined blast radius is within the budget for its tier.
2. **The verifier is never below the actor.** If the model that acts is stronger than the model that checks it, the check is decorative.

In [ ]:
TIER = {"Llama 3.2 1B (tiny)": 0, "Llama 3.3 8B (small)": 1,
        "GLM-4.6 (mid, open)": 2, "Kimi K2 (large, open)": 3}
TIER_BUDGET = {0: 0, 1: 3, 2: 20, 3: 60}

def routing_review(routes):
    problems = []
    for stage, (model, tools, gated) in routes.items():
        b = stage_blast(tools, gated)
        tier = TIER[model]
        if b > TIER_BUDGET[tier]:
            problems.append(f"{stage}: {model} (tier {tier}) holds blast {b} > "
                            f"budget {TIER_BUDGET[tier]}")
    actor_tier = max((TIER[m] for _, (m, t, g) in routes.items() if t), default=0)
    verifier_tier = TIER[routes["verify"][0]]
    if verifier_tier < actor_tier:
        problems.append(f"verifier tier {verifier_tier} < actor tier {actor_tier} "
                        f"— the check is weaker than the thing it checks")
    return problems

for label, r in (("as designed", ROUTES), ("fast-actor", opt1), ("cheap-verifier", opt2)):
    p = routing_review(r)
    print(f"{label:16s} {'PASS' if not p else 'FAIL'}")
    for x in p:
        print(f"                 ⚠ {x}")
assert not routing_review(ROUTES)

## What you just proved

The designed routing holds tools in one stage only, with a system blast of 3, and passes both rules. The fast-actor variant jumps to 43 and fails the tier budget. The cheap-verifier variant leaves the blast radius unchanged and fails the second rule — the point being that the metric alone would not have caught it.

## Your turn

Write your own routing table with the tier budgets your risk appetite implies, then check it against a real deployment. The verifier rule catches the most systems, and it is the one that never shows up in a cost review.

---

**Next → [A2.1 · "Who is calling?"](https://spbreed.github.io/cyber-commons/lessons/A2.1.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.8.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.8.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*